# 3. Improve the chatbot, one part at a time

In notebook 1 we built the chatbot from three parts we chose quickly:

| Part | What we used | Alternatives |
|---|---|---|
| **Parse** | pypdf: the characters in reading order | a layout-aware parser (LlamaParse), OCR for scanned pages |
| **Chunk** | fixed 200-word windows, one page at a time | cut at the document's own headings and keep tables whole; other sizes |
| **Search** | keywords (TF-IDF) | meaning (embeddings); both together ("hybrid") |

Notebook 2 gave us a way to score any version. Now we change **one part at a time** and score each version on the
same 24 questions, the way you would compare models in MLflow: same test set, one change per run.

| Setup | Parse | Chunk | Search |
|---|---|---|---|
| 1. baseline | PDF text | fixed | keywords |
| 2. better parsing | **Markdown** | fixed | keywords |
| 3. + chunk by structure | Markdown | **by headings** | keywords |
| 4. + search by meaning | Markdown | by headings | **embeddings** |

In [1]:
import os
import numpy as np
import pandas as pd

import rag
import rag_eval

pd.set_option("display.width", 200, "display.max_colwidth", None, "display.max_rows", 40)   # show full texts
assert os.environ.get("OPENROUTER_API_KEY"), "No key yet: open llm-cds/.env, paste the key after OPENROUTER_API_KEY=, save, restart the kernel"

SETUPS = {
    "1. baseline: PDF text, fixed, keywords": rag.Settings(docs="pdf", chunking="fixed", retriever="tfidf"),
    "2. Markdown, fixed, keywords":           rag.Settings(docs="markdown", chunking="fixed", retriever="tfidf"),
    "3. Markdown, by headings, keywords":     rag.Settings(docs="markdown", chunking="structure", retriever="tfidf"),
    "4. Markdown, by headings, embeddings":   rag.Settings(docs="markdown", chunking="structure", retriever="embed"),
}
results = {}

## 1. The baseline, again

Re-run notebook 2's baseline so all four setups are scored in the same session, by the same judge (about a minute).

In [2]:
name = "1. baseline: PDF text, fixed, keywords"
results[name] = rag_eval.evaluate_setup(SETUPS[name], name)

2026/10/28 22:08:48 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

Evaluating:   4%|▍         | 1/24 [Elapsed: 00:07, Remaining: 02:55]

Evaluating:   8%|▊         | 2/24 [Elapsed: 00:07, Remaining: 01:25]

Evaluating:  12%|█▎        | 3/24 [Elapsed: 00:08, Remaining: 00:59]

...

Evaluating:  92%|█████████▏| 22/24 [Elapsed: 00:23, Remaining: 00:02]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:28, Remaining: 00:00]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:28, Remaining: 00:00] [predict_fn: 17%, scorers: 83%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:28, Remaining: 00:00] [predict_fn: 17%, scorers: 83%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 1. baseline: PDF text, fixed, keywords
  Run ID: 7675392b1e0c4c5ea576eb0c6b61d017

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

1. baseline: PDF text, fixed, keywords: context_relevance 0.95, groundedness 1, answer_relevancy 0.84, refused_in_scope 9, refused_out_of_scope 4   (MLflow run 7675392b)


## 2. Better parsing: Markdown instead of PDF text

Notebook 1 showed one page both ways. The difference matters most for **tables**: in the PDF text, the cells of a
row are scattered, so a passage can hold "Doxycycline" and "300 mg" without saying which age group they belong to.

How much of the guidelines is in tables? Count the Markdown table rows:

In [3]:
for md_file in sorted(rag.DOCS_MD.glob("*.md")):
    lines = md_file.read_text().splitlines()
    table_rows = sum(line.startswith("|") for line in lines)
    print(f"{md_file.stem:35} {table_rows:5} table rows out of {len(lines):6} lines")

ephi_cebs_2021                        177 table rows out of   1144 lines
ephi_cholera_2022                     317 table rows out of   4375 lines
ephi_climate_sensitive_diseases_2022   150 table rows out of   2722 lines
ephi_emergency_nutrition_2022         407 table rows out of   2382 lines
moh_acute_malnutrition_2019          1352 table rows out of   5693 lines
moh_tb_2017                           753 table rows out of   6426 lines


Only the text changes; the chunking (200-word windows per page) and the search stay the same.

In [4]:
name = "2. Markdown, fixed, keywords"
results[name] = rag_eval.evaluate_setup(SETUPS[name], name)

2026/10/28 22:09:28 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

Evaluating:   4%|▍         | 1/24 [Elapsed: 00:09, Remaining: 03:30]

Evaluating:   8%|▊         | 2/24 [Elapsed: 00:09, Remaining: 01:43]

Evaluating:  12%|█▎        | 3/24 [Elapsed: 00:09, Remaining: 01:08]

...

Evaluating:  92%|█████████▏| 22/24 [Elapsed: 00:26, Remaining: 00:02]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:27, Remaining: 00:00]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:27, Remaining: 00:00] [predict_fn: 24%, scorers: 76%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:27, Remaining: 00:00] [predict_fn: 24%, scorers: 76%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 2. Markdown, fixed, keywords
  Run ID: c3f3e8d515a64647bc678f7de621bf8f

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

2. Markdown, fixed, keywords: context_relevance 0.91, groundedness 0.96, answer_relevancy 0.76, refused_in_scope 6, refused_out_of_scope 4   (MLflow run c3f3e8d5)


## 3. Chunk by structure: cut at the headings, keep tables whole

Fixed windows cut wherever the 200th word falls: in the middle of a table, or between a heading and the text under
it. The Markdown tells us where the document's own sections are (`#`, `##`, ...). **Structure chunking**:

- one passage per section (long sections are split; very short ones are merged with their neighbour);
- every passage starts with its **heading path**, e.g. `4. CHOLERA OUTBREAK RESPONSE > 4.2. Case Management >
  Antibiotic Treatment`, so a passage about "300 mg" also says what it is about;
- a table is **never cut**, even when it runs onto the next page (a passage can cover pages 37–38).

The cholera antibiotics page (38) again, now as structure passages:

In [5]:
structure = rag.make_structure_passages(rag.DOCS_MD)
fixed_md = rag.make_passages(rag.markdown_pages(rag.DOCS_MD))
print(f"fixed windows: {len(fixed_md)} passages;  by structure: {len(structure)} passages\n")

on_page_38 = structure[(structure.doc == "ephi_cholera_2022") & (structure.page <= 38) & (structure.end_page >= 38)]
for p in on_page_38.itertuples():
    print(f"--- {rag.page_label(p.page, p.end_page)}, {len(p.text.split())} words\n{p.text[:500]}\n")

fixed windows: 1508 passages;  by structure: 1187 passages

--- pages 37–38, 206 words
4. CHOLERA OUTBREAK RESPONSE > 4.2. Case Management > Monitoring of Patients with Severe Dehydration > Cholera with other infectious disease

The occurrence of new, emerging and re-emerging infectious diseases is challenge to human health. Deadly infectious diseases are one of the continues public health problems. Patients may have sign and symptoms unrelated to cholera for which the cause must be determined and appropriate treatment provided. During cholera epidemic, the deadly infectious disea

--- page 38, 207 words
4. CHOLERA OUTBREAK RESPONSE > 4.2. Case Management > Monitoring of Patients with Severe Dehydration > Antibiotic Treatment

* Doxycycline — single dose (300 mg for adults; 2–4 mg/kg for children under 12 years of age) is the antibiotic of choice for all patients, including pregnant women.

* If resistance to doxycycline is documented, give azithromycin 1 g or ciprofloxacin 1 g orally 

In [6]:
lengths = pd.DataFrame({"fixed windows": fixed_md.text.str.split().str.len().describe(),
                        "by structure": structure.text.str.split().str.len().describe()}).round()
lengths

,fixed windows,by structure
count,1508.0,1187.0
mean,159.0,189.0
std,53.0,104.0
min,21.0,24.0
25%,120.0,152.0
50%,200.0,182.0
75%,200.0,206.0
max,200.0,2167.0


> **PAUSE (2 min).** The longest structure passage is very long: what is it, probably? (Hint: a table is never
> cut.) What does a 2,000-word passage cost each time it is retrieved?

In [7]:
name = "3. Markdown, by headings, keywords"
results[name] = rag_eval.evaluate_setup(SETUPS[name], name)

2026/10/28 22:10:02 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

Evaluating:   4%|▍         | 1/24 [Elapsed: 00:04, Remaining: 01:54]

Evaluating:   8%|▊         | 2/24 [Elapsed: 00:06, Remaining: 01:10]

Evaluating:  12%|█▎        | 3/24 [Elapsed: 00:08, Remaining: 00:56]

...

Evaluating:  88%|████████▊ | 21/24 [Elapsed: 00:22, Remaining: 00:03]

Evaluating:  96%|█████████▌| 23/24 [Elapsed: 00:25, Remaining: 00:01]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:25, Remaining: 00:00] [predict_fn: 23%, scorers: 77%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:25, Remaining: 00:00] [predict_fn: 23%, scorers: 77%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 3. Markdown, by headings, keywords
  Run ID: 01886c0ea6704211837fd93625adbfe4

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

3. Markdown, by headings, keywords: context_relevance 1, groundedness 1, answer_relevancy 0.83, refused_in_scope 8, refused_out_of_scope 4   (MLflow run 01886c0e)


## 4. Search by meaning: embeddings

Keyword search only matches **the same words**. A health worker writes "loose stools", the guideline says
"diarrhoea"; "baby" versus "infant"; Amharic versus English.

An **embedding model** turns a text into a list of numbers (here 384) so that texts with similar **meaning** get
similar numbers, whatever the words. We use `BAAI/bge-small-en-v1.5`: small (67 MB), free, runs on a laptop.
Similarity is measured from 0 (unrelated) to 1 (same meaning):

In [8]:
from fastembed import TextEmbedding

embedder = TextEmbedding(rag.EMBED_MODEL, cache_dir=str(rag.INDEX_CACHE / "models"))


def similarity(a, b):
    va, vb = (np.array(v) for v in embedder.embed([a, b]))
    return float(va @ vb / np.linalg.norm(va) / np.linalg.norm(vb))


pairs = [("my baby has loose stools", "infant with diarrhoea"),
         ("my baby has loose stools", "adult with a cough for two weeks"),
         ("how much ORS", "volume of oral rehydration solution"),
         ("how much ORS", "how much does the vaccine cost")]
for a, b in pairs:
    print(f"{similarity(a, b):.2f}   {a!r}  vs  {b!r}")

0.81   'my baby has loose stools'  vs  'infant with diarrhoea'
0.59   'my baby has loose stools'  vs  'adult with a cough for two weeks'
0.58   'how much ORS'  vs  'volume of oral rehydration solution'
0.67   'how much ORS'  vs  'how much does the vaccine cost'


Each passage is embedded once (cached in `index/`); each question is embedded when asked, and the passages
whose numbers are closest win. Compare the two searches on the same question:

In [9]:
question = "My baby has loose stools and is very thirsty. What should I give?"
keywords = rag.build(SETUPS["3. Markdown, by headings, keywords"])
meaning = rag.build(SETUPS["4. Markdown, by headings, embeddings"])


def top5(retriever):
    hits = retriever.search(question, 5)
    return pd.DataFrame({"source": rag.source_ids(hits), "passage": hits.text.str.replace("\n", " ").tolist()})


side_by_side = pd.concat({"keywords": top5(keywords), "embeddings": top5(meaning)}, axis=1)
side_by_side.index = range(1, 6)
side_by_side.style.set_properties(**{"text-align": "left", "vertical-align": "top"})

> **Try it.** Change the question (an everyday phrasing, a misspelling, Amharic) and compare. When do keywords
> win? (Hint: exact drug names, codes, numbers.)

In [10]:
name = "4. Markdown, by headings, embeddings"
results[name] = rag_eval.evaluate_setup(SETUPS[name], name)

2026/10/28 22:10:33 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

Evaluating:   4%|▍         | 1/24 [Elapsed: 00:04, Remaining: 01:51]

Evaluating:   8%|▊         | 2/24 [Elapsed: 00:06, Remaining: 01:13]

Evaluating:  12%|█▎        | 3/24 [Elapsed: 00:06, Remaining: 00:47]

...

Evaluating:  92%|█████████▏| 22/24 [Elapsed: 00:22, Remaining: 00:02]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:24, Remaining: 00:00]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:24, Remaining: 00:00] [predict_fn: 22%, scorers: 78%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:24, Remaining: 00:00] [predict_fn: 22%, scorers: 78%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 4. Markdown, by headings, embeddings
  Run ID: d686fc1f655f4c258ad7ad037e3415af

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

4. Markdown, by headings, embeddings: context_relevance 1, groundedness 0.96, answer_relevancy 0.78, refused_in_scope 2, refused_out_of_scope 4   (MLflow run d686fc1f)


## 5. Compare

The same four summary numbers for every setup:

In [11]:
comparison = rag_eval.compare(results)
comparison

,context_relevance,groundedness,answer_relevancy,refused_in_scope,refused_out_of_scope
"1. baseline: PDF text, fixed, keywords",0.95,1.00,0.84,9.0,4.0
"2. Markdown, fixed, keywords",0.91,0.96,0.76,6.0,4.0
"3. Markdown, by headings, keywords",1.00,1.00,0.83,8.0,4.0
"4. Markdown, by headings, embeddings",1.00,0.96,0.78,2.0,4.0


### Guideline words vs everyday words

Half of the answerable questions use the guideline's own terms, half use everyday words (notebook 2). Split the
results by wording: the passages' relevance (over all 10 questions of each kind, answered or not) and how many the
assistant wrongly refused.

In [12]:
rag_eval.compare_wording(results)

context_relevance           wrong_refusals          
                                                everyday guideline       everyday guideline
1. baseline: PDF text, fixed, keywords              0.60      0.90        8 of 10   1 of 10
2. Markdown, fixed, keywords                        0.62      0.95        5 of 10   1 of 10
3. Markdown, by headings, keywords                  0.60      1.00        8 of 10   0 of 10
4. Markdown, by headings, embeddings                0.95      1.00        2 of 10   0 of 10

- **Questions in the guideline's words are solved by every setup**: context relevance 0.90–1.00, and from setup 3
  on, not one wrongly refused. Better parsing and chunking fixed the last few.
- **Questions in everyday words defeat keyword search, however good the parsing and chunking**: context relevance
  about 0.6 and 5–8 of 10 refused in setups 1–3. "Severe watery diarrhoea" doesn't share words with "cholera";
  "stop them getting sick too" doesn't share words with "isoniazid preventive therapy".
- **Embeddings search by meaning**: context relevance 0.95 and 2 of 10 refused on the same everyday questions.
- **So the best search depends on who asks.** Tested only with questions written in the guideline's words (as most
  test sets are, because experts write them), keywords look as good as embeddings. Real users don't write like the
  guideline. **The test questions must sound like the real users**, or the evaluation picks the wrong setup.
- **Some questions defeat both:** `sam_02` ("germ-killing medicine for a badly wasted toddler") and `sam_03`
  ("upper-arm measurement") are refused by every setup. Exercise 5 (both searches together) is one thing to try.

### Reading the summary

*These notes describe the run saved in this notebook. Your numbers will differ a little: the model and the judge
vary from run to run.*

- **Read the refusals before the averages.** The three scores are averaged over the questions each setup
  *answered*. Setup 3 shows a perfect 1.00 for context relevance and groundedness, but it refused 8 of the 20
  questions the guidelines answer: a setup that refuses every hard question looks perfect on the rest.
- **Wrong refusals:** 9, 6, 8, then **2** with embeddings. Each one is a health worker who got no answer although
  the guideline has one.
- **Groundedness and answer relevancy are a little lower for embeddings** (0.96 and 0.78). That is not worse answers
  to the same questions: setup 4 answers harder questions that the others refused, and some of those answers are
  only partly supported.

Why did embeddings make such a difference this time? The next table splits the questions by wording.

And question by question: where did each change help, and where did it hurt? (Groundedness and context
relevance; `R` = refused.)

In [13]:
def by_question(metric):
    table = {}
    for name, t in results.items():
        values = t.set_index("id")[metric].astype(float).round(2).astype(object)
        values[t.set_index("id").refused] = "R"
        table[name.split(":")[0]] = values             # e.g. "3. Markdown, by headings, keywords"
    return pd.DataFrame(table)


questions = rag_eval.load_questions().set_index("id")
by_question("context_relevance").join(questions.type)

,1. baseline,"2. Markdown, fixed, keywords","3. Markdown, by headings, keywords","4. Markdown, by headings, embeddings",type
id,,,,,
chol_01,1.0,0.75,R,1.0,answerable
chol_02,R,R,R,1.0,answerable
chol_03,R,0.5,1.0,1.0,answerable
chol_04,1.0,1.0,1.0,1.0,answerable
nutr_01,R,1.0,R,1.0,answerable
nutr_02,1.0,1.0,1.0,1.0,answerable
nutr_03,R,R,R,1.0,answerable
csd_01,1.0,1.0,1.0,1.0,answerable
csd_02,1.0,R,1.0,1.0,answerable


In [14]:
by_question("groundedness").join(questions.type)

,1. baseline,"2. Markdown, fixed, keywords","3. Markdown, by headings, keywords","4. Markdown, by headings, embeddings",type
id,,,,,
chol_01,1.0,1.0,R,0.75,answerable
chol_02,R,R,R,1.0,answerable
chol_03,R,1.0,1.0,1.0,answerable
chol_04,1.0,1.0,1.0,1.0,answerable
nutr_01,R,1.0,R,0.5,answerable
nutr_02,1.0,1.0,1.0,1.0,answerable
nutr_03,R,R,R,1.0,answerable
csd_01,1.0,1.0,1.0,1.0,answerable
csd_02,1.0,R,1.0,1.0,answerable


### Reading the question-by-question tables

- **Everyday-worded questions** (`chol_01`–`03`, `nutr_01`, `nutr_03`, `csd_03`, `cebs_02`, `sam_02`, `sam_03`,
  `tb_01`; see `questions.csv`) are where the `R`s are. `chol_02`, `nutr_03` and `cebs_02` are refused by every
  keyword setup and answered with embeddings.
- **Guideline-worded questions** are 1.0 almost everywhere.
- **`csd_03` is answered by every setup** although it is in everyday words: "sudden high fever and a stiff neck" are
  the guideline's words too. Everyday wording is a matter of degree.
- **A change that helps on average can hurt a question.** `chol_01` is answered by setups 1, 2 and 4 but refused by
  3; `csd_02` is refused only by setup 2. Before shipping a change, read the questions it made worse.

To read any one answer in full, with every passage the model saw: `explain(setup name, question id)`.

The example below is worth reading. `sam_01` asks about a **child in the outpatient programme (OTP)**. Read the
answer and passage [4]: which patients is that table about? In the run saved here the answer scores 1.0 on all
three checks, because every statement *is* in a passage. But the passage is the stabilisation-centre table for
**infants 0–6 months**, a different patient group. Grounded is not the same as right for this patient: this is
what the clinician's review of a sample is for.

In [15]:
def explain(name, question_id):
    row = results[name].set_index("id").loc[question_id]
    print(row.question, "\n\nANSWER:", row.answer, "\n")
    print("  ".join(f"{m} {row[m]:.2f}" if pd.notna(row[m]) else f"{m} -" for m in rag_eval.METRICS), row.judge_notes)
    for n, (source, text) in enumerate(zip(row.sources.split("; "), row.passages), 1):
        print(f"\n[{n}] {source}: {text}")


explain("4. Markdown, by headings, embeddings", "sam_01")

A child in the outpatient therapeutic programme still hasn't met the cure criteria. After how long is she classed as a non-responder, and when would a child instead be counted as a defaulter? 

ANSWER: A breastfed infant 0-6 months is classed as a non-responder if they do not reach the SAM discharge criteria after 16 weeks (4 months) in treatment [4]. A child is counted as a defaulter if they are absent for two consecutive days, and this default should be confirmed [4]. 

context_relevance 1.00  groundedness 1.00  answer_relevancy 0.86 

[1] moh_acute_malnutrition_2019:p75: 4 Management of Severe Acute Malnutrition (SAM) > 4.2 Stabilisation Centre (SC) for the Management of SAM With Medical Complications > 4.2.9 Treatment and Care of Infants 0-6 Months of Age

|Category|Definition|
|-|-|
|New admission|New case who meets the admission criteria for SC (see Table 23).|
|Readmission|Re-admitted for treatment in SC due to relapse or a returned defaulter who meets the admission criteria.|
|

In the **MLflow UI** (`mlflow ui` in `llm-cds/`): experiment **guideline-assistant**, tick the four runs, click
**Compare**. The parameters show exactly what changed between runs; the metrics show what it did. In **Traces**,
filter by run to see the same question answered by different setups.

> **Discussion (10 min).**
> 1. Which setup would you put in front of health workers, and why? Which number did you look at first?
> 2. Did any change make some questions **worse**? Would you have noticed from the averages alone?
> 3. Differences of 0.02 or 0.03 on 20 questions: real, or noise? (Run one setup twice and see how much it moves.)
> 4. What did each change cost? The LlamaParse step is paid, once; embeddings take minutes on a laptop, once;
>    longer passages cost tokens on every question.
> 5. What would you record so that in six months someone can tell **which version** answered a given question?

## Exercises

Each exercise is one cell: change the line marked `# <- CHANGE`, run the cell (about a minute), and read the new
row at the bottom of the comparison table. The setup it starts from is **3. Markdown, by headings, keywords**, the
best of the four above. Each run also appears in MLflow, next to the others.

**Exercise 1. How many passages?** The model now gets 5 passages (`k=5`). Fewer means less to read and lower
cost; more means more chances to include the answer, and more to get lost in. Try `k=2`, then `k=10`.

In [ ]:
k = 10                                                       # <- CHANGE: try 2, then 10 (the setups above used 5)

name = f"ex1. k={k}"
results[name] = rag_eval.evaluate_setup(
    rag.Settings(docs="markdown", chunking="structure", retriever="tfidf", k=k), name)
rag_eval.compare(results)

**Exercise 2. Passage size.** Back to fixed windows on the Markdown (setup 2), with a different window size.
Small windows are precise but lose context; large ones keep context but dilute the search. Try 100 and 400 words.

In [ ]:
words = 400                                                  # <- CHANGE: try 100, then 400 (setup 2 used 200)

name = f"ex2. fixed windows of {words} words"
results[name] = rag_eval.evaluate_setup(
    rag.Settings(docs="markdown", chunking="fixed", retriever="tfidf", chunk_words=words, overlap_words=words // 5), name)
rag_eval.compare(results)

**Exercise 3. A smaller model.** The same search, a smaller and cheaper LLM to write the answers. How much worse
is it, if at all? Compare with setup 3.

In [ ]:
model = "qwen/qwen3.5-9b"                                     # <- CHANGE: a 9-billion-parameter model (Gemma 4 is 26b)

name = f"ex3. {model}"
results[name] = rag_eval.evaluate_setup(
    rag.Settings(docs="markdown", chunking="structure", retriever="tfidf", model=model), name)
rag_eval.compare(results)

**Exercise 4. The prompt.** Add one instruction to the prompt and save it as a new version, `v2`. A new prompt is a
new version of the assistant, so it gets its own name, and MLflow records which one each run used.

In [ ]:
new_rule = "- If the passages answer only part of the question, answer that part and say which part is missing."
                                                             # <- CHANGE: write your own rule (one line, starting with "- ")

rag.PROMPTS["v2"] = rag.PROMPTS["v1"] + "\n" + new_rule
print(rag.PROMPTS["v2"], "\n")

name = "ex4. prompt v2"
results[name] = rag_eval.evaluate_setup(
    rag.Settings(docs="markdown", chunking="structure", retriever="tfidf", prompt_version="v2"), name)
rag_eval.compare(results)

**Exercise 5 (harder). Both searches together.** Take some passages from keyword search and some from embedding
search ("hybrid" search). The code is complete; change how many come from each.

In [ ]:
from_keywords = 3                                            # <- CHANGE: how many of the 5 passages come from keywords
from_meaning = 5 - from_keywords                             #    the rest come from embeddings


class Hybrid:
    """The best passages of keyword search, then the best of embedding search that aren't already in."""

    def __init__(self, keywords, meaning):
        self.keywords, self.meaning, self.passages = keywords, meaning, keywords.passages

    def search(self, question, k=5):
        first = self.keywords.search(question, from_keywords)
        extra = self.meaning.search(question, k)
        extra = extra[~extra.text.isin(first.text)].head(k - len(first))
        return pd.concat([first, extra])


name = f"ex5. hybrid: {from_keywords} keywords + {from_meaning} embeddings"
results[name] = rag_eval.evaluate_setup(
    rag.Settings(docs="markdown", chunking="structure", retriever="hybrid"), name, retriever=Hybrid(keywords, meaning))
rag_eval.compare(results)

After the exercises, `by_question("groundedness")` and `by_question("context_relevance")` show your runs next to
the four setups, and `explain(name, question_id)` shows any answer in full.

> **Share (5 min).** Each pair reports one change that helped and one that didn't. Did anyone find a setup that
> beats setup 3 by more than the noise?